# Notebook 7 — PD Calibration (Basel IRB / IFRS 9)
## Mortgage Credit Risk Modelling

This notebook explores the calibration analysis from `08_calibration.py`.

**Why calibration is separate from discrimination:**

A model can rank-order borrowers perfectly (AUROC = 1.0) while being completely wrong about the *magnitude* of risk. For ECL calculations, `PD = 5%` vs `PD = 0.5%` makes a 10× difference in provisioning — getting the level right is as important as getting the ordering right.

**Methods compared:**
- **Platt scaling** — 1D logistic regression on raw scores: `P_cal = σ(a·s + b)`
- **Isotonic regression** — non-parametric monotone mapping
- **Temperature scaling** — single-parameter: `P_cal = σ(logit(P_raw) / T)`

**Prerequisites:** Run `02_pd_logistic_regression.py`, `03_pd_ensemble.py`, then `08_calibration.py`.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

plt.rcParams.update({
    "figure.facecolor": "#0F1117", "axes.facecolor": "#0F1117",
    "axes.edgecolor":   "#2D3748", "axes.labelcolor": "#E2E8F0",
    "xtick.color":      "#A0AEC0", "ytick.color":     "#A0AEC0",
    "text.color":       "#E2E8F0", "grid.color":      "#1A2035",
    "legend.facecolor": "#1A2035", "legend.edgecolor":"#2D3748",
    "font.family": "monospace",    "figure.dpi": 120,
})
C = {"sky":"#38BDF8","gold":"#FBBF24","green":"#34D399","red":"#F87171","purple":"#A78BFA"}
PROC = Path("data/processed")
FIG  = Path("data/figures")
print("✓ Environment ready")

needed = ["calibration_metrics.csv"]
missing = [f for f in needed if not (PROC/f).exists()]
if missing:
    print("⚠  Missing:", missing)
else:
    cal_df = pd.read_csv(PROC/"calibration_metrics.csv")
    print("Calibration metrics loaded:")
    print(cal_df.to_string(index=False))


## 1. Calibration Metrics — Before vs After

Lower Brier score and ECE = better calibration.  
The Hosmer-Lemeshow p-value > 0.05 indicates acceptable calibration (fail to reject H₀: model is well-calibrated).


In [ ]:
if (PROC/"calibration_metrics.csv").exists():
    cal = pd.read_csv(PROC/"calibration_metrics.csv")
    disp_cols = [c for c in ["label","brier","ece","mce","bias","hl_pval"] if c in cal.columns]
    if disp_cols:
        print("=" * 80)
        print(cal[disp_cols].to_string(index=False))
        print("=" * 80)

    # Bar chart: Brier and ECE side by side
    metric_cols = [c for c in ["brier","ece"] if c in cal.columns]
    if metric_cols and "label" in cal.columns:
        fig, axes = plt.subplots(1, len(metric_cols), figsize=(13, 5))
        if len(metric_cols) == 1: axes = [axes]
        fig.suptitle("Calibration Improvement — Before vs After  (OOS Evaluation Set)\n"
                     "Lower Brier Score and ECE = better calibration",
                     fontsize=12, fontweight="bold", color="white")

        palette_map = {"Raw":"#F59E0B","Platt":"#38BDF8",
                       "Isotonic":"#10B981","Temperature":"#A78BFA"}
        ylabel_map  = {"brier":"Brier Score  (lower = better)",
                       "ece":  "Expected Calibration Error  (lower = better)"}

        for ax, metric in zip(axes, metric_cols):
            labels = cal["label"].tolist()
            values = cal[metric].tolist()
            colors = []
            for lbl in labels:
                matched = next((c for k,c in palette_map.items() if k in lbl), "#E2E8F0")
                colors.append(matched)
            bars = ax.bar(range(len(labels)), values, color=colors,
                          alpha=0.85, edgecolor="none", width=0.6)
            for bar, val in zip(bars, values):
                if val == val:  # not NaN
                    ax.text(bar.get_x()+bar.get_width()/2,
                            bar.get_height()+max(v for v in values if v==v)*0.01,
                            f"{val:.6f}", ha="center", va="bottom",
                            fontsize=8.5, color="#E2E8F0")
            ax.set_xticks(range(len(labels)))
            ax.set_xticklabels(labels, rotation=22, ha="right", fontsize=9)
            ax.set_ylabel(ylabel_map.get(metric, metric), fontsize=10)
            ax.grid(True, axis="y", alpha=0.25)
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        plt.tight_layout(); plt.show()


## 2. Reliability Diagrams

A perfectly calibrated model has all points on the 45° diagonal. Points above the line = model under-predicts risk (too conservative). Points below = over-predicts (too aggressive).

**Interpreting the histogram** at the bottom: if most loans are scored near zero with a small tail of high-PD loans, the model is "confident" — which is fine as long as those high-PD predictions verify against observed default rates.


In [ ]:
from IPython.display import Image, display

rel_figs = [
    ("calibration_reliability_lr.png",  "Reliability Diagram — Logistic Regression"),
    ("calibration_reliability_xgb.png", "Reliability Diagram — XGBoost"),
    ("calibration_comparison.png",      "Brier / ECE Comparison — All Methods"),
    ("calibration_lradr.png",           "LRADR — Calibrated PD vs Observed Default Rate"),
]
for fname, caption in rel_figs:
    path = FIG / fname
    if path.exists():
        print(f"\n{'─'*60}\n  {caption}\n{'─'*60}")
        display(Image(str(path), width=820))
    else:
        print(f"  ⚠  {fname} — run 08_calibration.py first")


## 3. LRADR — Long-Run Average Default Rate

Basel II §461 requires that calibrated PD estimates approximate the long-run average default rate over an economic cycle. This check compares mean predicted PD (across calibration methods) against the observed default rate on OOS and OOT sets.

A well-calibrated model should have bias ≈ 0 on the OOS set. OOT bias may be larger due to the post-2017 rate environment shift — this is expected and disclosed, not a model failure.


In [ ]:
if (PROC/"calibrated_scores_oos.csv").exists():
    cal_scores = pd.read_csv(PROC/"calibrated_scores_oos.csv")
    print("Calibrated OOS scores — first 5 rows:")
    print(cal_scores.head().to_string(index=False))
    print(f"\nTotal rows: {len(cal_scores):,}")

    # Bias summary
    score_cols = [c for c in cal_scores.columns
                  if any(m in c for m in ["platt","isotonic","temperature","score","xgb"])
                  and c != "default_12m"]
    target_col = "default_12m"
    if target_col in cal_scores.columns and score_cols:
        obs_rate = cal_scores[target_col].mean() * 100
        print(f"\n  Observed default rate (OOS eval): {obs_rate:.4f}%")
        print(f"  {'Method':<25}  {'Mean PD (%)':>12}  {'Bias (pp)':>12}  {'Calibrated?'}")
        print("  " + "─" * 65)
        for col in score_cols:
            if col in cal_scores.columns:
                mean_pd = cal_scores[col].mean() * 100
                bias    = mean_pd - obs_rate
                ok      = "✓" if abs(bias) < 0.1 else "⚠"
                print(f"  {col:<25}  {mean_pd:>12.4f}  {bias:>+12.4f}  {ok}")


## Summary

| Method | Pros | Cons | Best for |
|---|---|---|---|
| Raw (uncalibrated) | No extra step | Inflated by class reweighting | Rank-ordering only |
| Platt scaling | Stable, monotone | Assumes sigmoid shape | Small calibration sets |
| Isotonic regression | Flexible, non-parametric | Needs 500+ events | Large datasets |
| Temperature scaling | Single interpretable parameter | Less flexible | When over-confidence is the problem |

**Recommendation for regulatory use:** Platt scaling on a dedicated held-out calibration set (not the OOS evaluation set) — it is stable, auditable (two parameters: `a` and `b`), and preserves rank order.

The calibrated PD from `calibrated_scores_oos.csv` feeds directly into:
```
ECL = PD_calibrated × LGD × EAD
```
